In [ ]:
# EDA -- Dataset 2: Apache Jira (Parquet) -- ShiftMetrics Analytics [CORREGIDO]
# Colecciones: issues, events, comments, worklogs, projects, users
# Fixes:
#   1. Clasificador de colecciones: regla manual + ISSUE_COLS ampliado
#   2. Transiciones: extraidas de events.items (no coleccion separada)
#   3. Cycle Time: calculado tanto desde issues.resolutiondate como desde events
#   4. Muestra ampliada: multiples archivos por coleccion
#   5. Bug-to-Story Ratio calculado
#   6. Extraccion de proyecto desde campo key (HADOOP-1234 -> HADOOP)


In [ ]:
# -- 0. Autenticacion GCP + Imports
from google.colab import auth
auth.authenticate_user()
print('OK Autenticado')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import storage
import io, os, json as _json
from collections import defaultdict

pd.set_option('display.max_columns', 50)
sns.set_theme(style='darkgrid')

PROJECT = 'shiftmetrics-analytics'
BUCKET  = 'shiftmetrics-bronze'
PREFIX  = 'apache-jira-parquet/'
print('OK Imports OK')


In [ ]:
# -- 1. Listar colecciones Parquet
client = storage.Client(project=PROJECT)
bucket_obj = client.bucket(BUCKET)

blobs = list(bucket_obj.list_blobs(prefix=PREFIX))
parquet_blobs = [b for b in blobs if b.name.endswith('.parquet') or '.snappy' in b.name]

collections = defaultdict(list)
for b in parquet_blobs:
    parts = b.name.replace(PREFIX, '').split('/')
    coll = parts[0] if len(parts) > 1 else 'root'
    collections[coll].append(b)

print('Colecciones encontradas:', len(collections))
for coll, blbs in sorted(collections.items()):
    total_mb = sum(b.size for b in blbs) / (1024**2)
    print(f'  {coll:20s}  {len(blbs):5d} archivos  {total_mb:.1f} MB')


In [ ]:
# -- 2. Cargar muestra AMPLIADA de cada coleccion (hasta 5 archivos)
# CORRECCION: antes se cargaba solo 1 archivo por coleccion (muestra insuficiente)
SAMPLE_FILES = 5  # archivos por coleccion
coll_dfs = {}

for coll_name, blbs in sorted(collections.items()):
    try:
        frames = []
        for blob in blbs[:SAMPLE_FILES]:
            data = blob.download_as_bytes()
            df = pd.read_parquet(io.BytesIO(data))
            frames.append(df)
        combined = pd.concat(frames, ignore_index=True)
        coll_dfs[coll_name] = combined
        print(f'OK {coll_name:25s}  {combined.shape[0]:8,} filas x {combined.shape[1]:3d} cols  ({len(frames)} archivos)')
    except Exception as e:
        print(f'ERR {coll_name}: {e}')

print(f'\nColecciones cargadas: {len(coll_dfs)}')


In [ ]:
# -- 3. Clasificar colecciones
# CORRECCION: regla manual para issues + ISSUE_COLS ampliado con key, summary
print('=== ANALISIS DE COLUMNAS POR COLECCION ===\n')

ISSUE_COLS = ['issuetype','issue_type','type','IssueType','issueType',
              'issueid','issue_id','key','summary','priority']
DATE_COLS  = ['created','updated','resolved','resolutiondate','duedate','Created','Updated']

coll_types = {}
for coll_name, df in coll_dfs.items():
    cols_lower = {c.lower() for c in df.columns}
    has_issues = any(c.lower() in cols_lower for c in ISSUE_COLS)
    has_dates  = any(c.lower() in cols_lower for c in DATE_COLS)

    # Regla manual: la coleccion llamada 'issues' SIEMPRE es tipo issues
    if coll_name == 'issues':
        coll_type = 'issues'
    elif has_issues:
        coll_type = 'issues'
    else:
        coll_type = 'other'

    coll_types[coll_name] = coll_type
    date_found = [c for c in df.columns if c.lower() in {d.lower() for d in DATE_COLS}]
    print(f'{coll_name} -> {coll_type.upper()}')
    print(f'   Cols ({len(df.columns)}): {list(df.columns)[:10]}')
    if date_found: print(f'   Fechas: {date_found}')
    print()

issues_colls = [k for k,v in coll_types.items() if v == 'issues']
print('Colecciones ISSUES:', issues_colls)


In [ ]:
# -- 4. EDA coleccion Issues: tipos, proyectos, Bug-to-Story Ratio
# NUEVO: Bug-to-Story Ratio y extraccion de proyecto desde key
issues_df = coll_dfs.get('issues')

if issues_df is not None:
    print(f'issues shape: {issues_df.shape}')
    print(f'Columnas: {list(issues_df.columns)}')

    # -- Tipo de issue
    type_col = None
    for tc in ['issuetype','issue_type','type','IssueType']:
        if tc in issues_df.columns:
            type_col = tc
            break

    if type_col:
        print(f'\nColumna tipo: {type_col}')
        vc = issues_df[type_col].value_counts()
        print('Distribucion de tipos:')
        print(vc.head(20).to_string())

        # Bug-to-Story Ratio
        bug_count   = vc.get('Bug', vc.get('bug', 0))
        story_count = vc.get('Story', vc.get('story', 0))
        task_count  = vc.get('Task', vc.get('task', 0))
        total_issues = vc.sum()
        bsr = bug_count / (story_count + task_count) if (story_count + task_count) > 0 else None
        print(f'\nBug-to-Story Ratio = {bug_count} bugs / ({story_count} stories + {task_count} tasks) = {bsr:.3f}' if bsr else 'BSR: no se puede calcular')
        print(f'Proporcion bugs/total: {bug_count/total_issues*100:.1f}%' if total_issues > 0 else '')

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        top8 = vc.head(8)
        ax1.pie(top8.values, labels=top8.index, autopct='%1.1f%%', startangle=90)
        ax1.set_title('Tipos de issue (Apache Jira)')
        vc.head(15).plot(kind='barh', ax=ax2, color='#3498db', edgecolor='black')
        ax2.set_title('Top 15 tipos')
        ax2.invert_yaxis()
        plt.tight_layout()
        plt.savefig('apache_jira_issue_types.png', dpi=120)
        plt.show()
    else:
        print('WARN No se detecto columna de tipo. Cols:', list(issues_df.columns))

    # -- Extraccion de proyecto desde key (HADOOP-1234 -> HADOOP)
    if 'key' in issues_df.columns:
        issues_df['proyecto'] = issues_df['key'].str.split('-').str[0]
        print('\nProyectos en muestra:')
        print(issues_df['proyecto'].value_counts().head(20).to_string())
    else:
        print('WARN Columna key no encontrada')
else:
    print('ERR Coleccion issues no cargada')


In [ ]:
# -- 5. Completitud de campos por coleccion
fig, axes = plt.subplots(1, len(coll_dfs), figsize=(5*len(coll_dfs), 6))
if len(coll_dfs) == 1: axes = [axes]

for ax, (coll_name, df) in zip(axes, coll_dfs.items()):
    null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=True)
    null_pct.plot(kind='barh', ax=ax, color='#e74c3c', alpha=0.7)
    ax.set_title(f'{coll_name}\n({len(df):,} filas)')
    ax.set_xlabel('% nulos')
    ax.axvline(50, color='orange', ls='--', lw=1, label='50%')
    ax.legend(fontsize=8)

plt.suptitle('Completitud por coleccion -- Apache Jira', fontsize=14)
plt.tight_layout()
plt.savefig('apache_jira_nulls.png', dpi=120)
plt.show()

print('=== COLUMNAS CON > 50% NULOS POR COLECCION ===')
for coll_name, df in coll_dfs.items():
    bad = (df.isnull().sum() / len(df) * 100)
    bad = bad[bad > 50]
    print(f'\n{coll_name}: {len(bad)} cols con >50% nulos')
    if len(bad) > 0: print(bad.to_string())


In [ ]:
# -- 6. Cycle Time DIRECTO desde issues (resolutiondate - created)
# NUEVO: calcular CT directamente desde issues, sin depender de events
if issues_df is not None:
    date_pairs = [('created','resolutiondate'), ('created','resolved'),
                  ('Created','Resolved'), ('createdDate','resolutionDate')]
    ct_direct = None
    for start_col, end_col in date_pairs:
        if start_col in issues_df.columns and end_col in issues_df.columns:
            t_start = pd.to_datetime(issues_df[start_col], errors='coerce')
            t_end   = pd.to_datetime(issues_df[end_col],   errors='coerce')
            ct = (t_end - t_start).dt.days
            ct_valid = ct.dropna()
            ct_valid = ct_valid[ct_valid >= 0]
            if len(ct_valid) > 0:
                ct_direct = ct_valid
                print(f'Cycle Time desde {start_col} -> {end_col}')
                print(f'Issues con resolucion: {len(ct_valid):,} / {len(issues_df):,}')
                print(ct_valid.describe().to_string())
                fig, ax = plt.subplots(figsize=(10, 4))
                ct_valid.clip(0, 500).hist(bins=50, ax=ax, color='#9b59b6', edgecolor='black', alpha=0.8)
                ax.axvline(ct_valid.median(), color='red', ls='--', label=f'Mediana: {ct_valid.median():.0f} dias')
                ax.set_title('Cycle Time directo desde issues (Apache Jira)')
                ax.set_xlabel('Dias')
                ax.legend()
                plt.tight_layout()
                plt.savefig('apache_jira_ct_direct.png', dpi=120)
                plt.show()
                break
    if ct_direct is None:
        print('WARN No se pudo calcular CT directo. Cols de issues:', list(issues_df.columns)[:20])
else:
    print('ERR Coleccion issues no cargada')


In [ ]:
# -- 7. Cycle Time desde events.items (transiciones de estado)
# CORRECCION BLOCKER 2: las transiciones NO estan en coleccion separada
# Estan en events.items: lista de dicts con field, fromString, toString
print('=== EXTRAYENDO TRANSICIONES DE events.items ===\n')

events_df = coll_dfs.get('events')

if events_df is None:
    print('ERR Coleccion events no cargada')
else:
    print(f'events shape: {events_df.shape}')
    print(f'Columnas: {list(events_df.columns)}')
    print(f'\nEjemplo items[0]: {events_df["items"].iloc[0]}')

    rows = []
    for _, ev in events_df.iterrows():
        items = ev.get('items')
        if items is None: continue
        if isinstance(items, str):
            try: items = _json.loads(items)
            except: continue
        if not isinstance(items, (list, tuple)): items = [items]
        for it in items:
            if not isinstance(it, dict): continue
            if str(it.get('field','')).lower() == 'status':
                rows.append({
                    'issue':       ev.get('issue'),
                    'event_date':  ev.get('created'),
                    'from_status': it.get('fromString', it.get('from','')),
                    'to_status':   it.get('toString',   it.get('to',  '')),
                })

    trans_df = pd.DataFrame(rows)
    print(f'\nTransiciones de estado extraidas: {len(trans_df):,} filas')

    if len(trans_df) > 0:
        print('\nDistribucion to_status:')
        print(trans_df['to_status'].value_counts().head(15).to_string())

        trans_df['event_date'] = pd.to_datetime(trans_df['event_date'], errors='coerce')

        ACTIVE_STATUSES = {'in progress','in review','patch available','coding in progress'}
        CLOSED_STATUSES = {'resolved','closed','done','fixed'}

        def get_first_date(group, statuses):
            mask = group['to_status'].str.lower().isin(statuses)
            sub  = group[mask].sort_values('event_date')
            return sub['event_date'].iloc[0] if len(sub) > 0 else pd.NaT

        cycle_rows = []
        for issue_id, grp in trans_df.groupby('issue'):
            t_active = get_first_date(grp, ACTIVE_STATUSES)
            t_closed = get_first_date(grp, CLOSED_STATUSES)
            if pd.notna(t_active) and pd.notna(t_closed) and t_closed > t_active:
                cycle_rows.append({'issue': issue_id,
                                   'cycle_time_days': (t_closed - t_active).days})

        ct_events_df = pd.DataFrame(cycle_rows)
        print(f'\nIssues con CT calculable via events: {len(ct_events_df):,}')

        if len(ct_events_df) > 0:
            print(ct_events_df['cycle_time_days'].describe().to_string())
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            ct_events_df['cycle_time_days'].clip(0,200).hist(bins=40, ax=axes[0],
                color='#3498db', edgecolor='black', alpha=0.8)
            axes[0].set_title('Cycle Time via events.items (Apache Jira)')
            axes[0].set_xlabel('Dias')
            axes[0].axvline(ct_events_df['cycle_time_days'].median(), color='red', ls='--',
                label=f'Mediana: {ct_events_df["cycle_time_days"].median():.0f}d')
            axes[0].legend()
            top_trans = trans_df.groupby(['from_status','to_status']).size().sort_values(ascending=False).head(10)
            top_trans.plot(kind='barh', ax=axes[1], color='#2ecc71', edgecolor='black')
            axes[1].set_title('Top 10 transiciones de estado')
            axes[1].invert_yaxis()
            plt.tight_layout()
            plt.savefig('apache_jira_cycle_time_events.png', dpi=120)
            plt.show()
        else:
            print('Muestra pequeña: no hay pares In Progress->Resolved en estos archivos')
            print('La extraccion de transiciones SI funciona. Estados unicos:', trans_df['to_status'].nunique())
    else:
        print('No se encontraron transiciones de status en items de esta muestra')


In [ ]:
# -- 8. Distribucion temporal
if issues_df is not None:
    DATE_CANDIDATES = ['created','createdDate','Created','creation_date']
    date_col = next((c for c in DATE_CANDIDATES if c in issues_df.columns), None)
    if date_col:
        try:
            dates = pd.to_datetime(issues_df[date_col], errors='coerce').dropna()
            print(f'Fecha: {date_col} | Rango: {dates.min()} -> {dates.max()}')
            print('Issues por ano:')
            print(dates.dt.year.value_counts().sort_index().to_string())
            fig, ax = plt.subplots(figsize=(12, 4))
            dates.dt.to_period('M').value_counts().sort_index().plot(ax=ax, color='#2ecc71')
            ax.set_title('Issues por mes -- Apache Jira')
            ax.set_xlabel('Mes'); ax.set_ylabel('Issues')
            plt.tight_layout()
            plt.savefig('apache_jira_timeline.png', dpi=120)
            plt.show()
        except Exception as e:
            print(f'WARN Error fechas: {e}')
    else:
        print('WARN No se encontro columna de fecha. Cols:', list(issues_df.columns)[:15])
else:
    print('ERR Coleccion issues no cargada')


In [ ]:
# -- 9. Resumen EDA -- parametros para Silver
print('=' * 65)
print('RESUMEN EDA -- APACHE JIRA (Parquet) [CORREGIDO]')
print('=' * 65)
print('Colecciones encontradas:', len(collections))
print('Colecciones cargadas:', len(coll_dfs))

for coll_name, coll_type in coll_types.items():
    df = coll_dfs.get(coll_name)
    if df is not None:
        null_pct = (df.isnull().sum() / len(df) * 100)
        good = null_pct[null_pct < 20]
        print(f'  {coll_name} [{coll_type}]: {len(df):,} filas | {len(good)}/{len(df.columns)} cols completas')

if issues_df is not None and type_col:
    vc = issues_df[type_col].value_counts()
    bug_c  = vc.get('Bug', vc.get('bug', 0))
    story_c= vc.get('Story', vc.get('story', 0))
    task_c = vc.get('Task', vc.get('task', 0))
    bsr    = bug_c / (story_c + task_c) if (story_c + task_c) > 0 else None
    print(f'\nBug-to-Story Ratio observado: {bsr:.3f}' if bsr else 'BSR: no calculable en muestra')
    print(f'Proporcion bugs: {bug_c/vc.sum()*100:.1f}%' if vc.sum() > 0 else '')

print('\n--- PARAMETROS PARA JOB SILVER APACHE JIRA ---')
print('Colecciones a usar: issues, events')
print('Colecciones a descartar: comments, worklogs, users (no aportan a ML)')
print('Campos clave issues: id, key, issuetype, status, priority, created, resolutiondate')
print('Cols a dropear (>50% nulos): regression, patchinfo, environment, duedate, timeestimate')
print('Cycle Time: resolutiondate - created (en dias)')
print('Proyecto: extraer prefijo de key (HADOOP-1234 -> HADOOP)')
print('Transiciones: explotar events.items, filtrar field=status')
print('Particion Parquet Silver: por proyecto')

print('--- PARAMETROS PARA DATASET 5 (SIMULADOR) ---')
print('bug_to_story_ratio: valor BSR observado arriba')
print('cycle_time_dias: median del CT directo')
print('issues_por_sprint: extraer de agrupacion por proyecto/mes')

print('OK EDA Apache Jira completado y corregido')
